<a href="https://colab.research.google.com/github/Bilra-Hkr/BigData26_B_2411533007_DerielChaerahman/blob/main/BD_P06_2411533007_DerielChaerahman.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# D. Persiapan Praktikum

In [1]:
!pip install pyspark

In [6]:
from pyspark.sql import SparkSession

# Inisialisasi SparkSession dan cek konfigurasi
spark = SparkSession.builder \
    .appName("PraktikumBigData") \
    .master("local[*]") \
    .getOrCreate()

print("Versi Spark:", spark.version)


Versi Spark: 4.0.5


In [7]:
print("Spark Application Name:", spark.sparkContext.appName)
print("Spark Master:", spark.sparkContext.master)

Spark Application Name: PraktikumBigData
Spark Master: local[*]


- Memberi nama pada proyek/aplikasi
- instruksi kepada Spark untuk berjalan di komputer lokal dengan mengerahkan seluruh core CPU (*) yang tersedia sebagai pekerja.

# E.1. Mengenal RDD pada Apache Spark

In [8]:
# Membuat RDD
data = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

rdd = spark.sparkContext.parallelize(data)

print("Jumlah elemen:", rdd.count())

Jumlah elemen: 10


In [11]:
# Melihat isi RDD
print(rdd.collect())

[10, 20, 30, 40, 50, 60, 70, 80, 90, 100]


collect() mengambil seluruh elemen dari RDD yang tersebar di berbagai worker/executor dan dikembalikan sebagai list data.

karena collect() memaksa spark untuk mengeksekusi seluruh rangkaian transformation yang tertunda, dan mengembalikan output pemrosesan ke driver program.

In [12]:
# Menerapkan Transformation menggunakan map() dan setiap elemen di kali 2
rdd_hasil = rdd.map(lambda x: x * 2)

print (rdd_hasil.collect())

[20, 40, 60, 80, 100, 120, 140, 160, 180, 200]


In [14]:
# Menerapkan filter untuk elemen > 50
rdd_filter = rdd.filter(lambda x: x > 50)

print(rdd_filter.collect())

[60, 70, 80, 90, 100]


In [16]:
# Menggabungkan beberapa Transformation
rdd_genap = rdd.filter(lambda x: x % 2 == 0)
rdd_kali = rdd_genap.map(lambda x: x * 3)

print(rdd_kali.collect())

[30, 60, 90, 120, 150, 180, 210, 240, 270, 300]


# E.2 Membuat dan mengolah dataFrame

In [17]:
# Membuat dataset
data_transaksi = [
    (1, "Laptop", "Elektronik", 7500000, 2),
    (2, "Mouse", "Aksesoris", 150000, 5),
    (3, "Keyboard", "Aksesoris", 300000, 3),
    (4, "Monitor", "Elektronik", 2500000, 2),
    (5, "Flashdisk", "Aksesoris", 80000, 10),
    (6, "Printer", "Elektronik", 1800000, 1),
    (7, "Headset", "Aksesoris", 250000, 4),
    (8, "Laptop", "Elektronik", 7500000, 1),
    (9, "Mouse", "Aksesoris", 150000, 8),
    (10, "Monitor", "Elektronik", 2500000, 3)
]

kolom = ["id_transaksi", "produk", "kategori", "harga", "jumlah"]
df = spark.createDataFrame(data_transaksi, kolom)

In [18]:
# Menampilkan data

df.show()

+------------+---------+----------+-------+------+
|id_transaksi|   produk|  kategori|  harga|jumlah|
+------------+---------+----------+-------+------+
|           1|   Laptop|Elektronik|7500000|     2|
|           2|    Mouse| Aksesoris| 150000|     5|
|           3| Keyboard| Aksesoris| 300000|     3|
|           4|  Monitor|Elektronik|2500000|     2|
|           5|Flashdisk| Aksesoris|  80000|    10|
|           6|  Printer|Elektronik|1800000|     1|
|           7|  Headset| Aksesoris| 250000|     4|
|           8|   Laptop|Elektronik|7500000|     1|
|           9|    Mouse| Aksesoris| 150000|     8|
|          10|  Monitor|Elektronik|2500000|     3|
+------------+---------+----------+-------+------+



In [19]:
# Melihat strutkur Dataframe

df.printSchema()

root
 |-- id_transaksi: long (nullable = true)
 |-- produk: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- jumlah: long (nullable = true)



In [20]:
print("Jumlah transaksi:", df.count())

Jumlah transaksi: 10


In [21]:
df.select(
    "Produk",
    "kategori",
    "harga"
) .show()

+---------+----------+-------+
|   Produk|  kategori|  harga|
+---------+----------+-------+
|   Laptop|Elektronik|7500000|
|    Mouse| Aksesoris| 150000|
| Keyboard| Aksesoris| 300000|
|  Monitor|Elektronik|2500000|
|Flashdisk| Aksesoris|  80000|
|  Printer|Elektronik|1800000|
|  Headset| Aksesoris| 250000|
|   Laptop|Elektronik|7500000|
|    Mouse| Aksesoris| 150000|
|  Monitor|Elektronik|2500000|
+---------+----------+-------+



In [22]:
# filtering
df.filter(
    df.harga > 1000000
) .show()

+------------+-------+----------+-------+------+
|id_transaksi| produk|  kategori|  harga|jumlah|
+------------+-------+----------+-------+------+
|           1| Laptop|Elektronik|7500000|     2|
|           4|Monitor|Elektronik|2500000|     2|
|           6|Printer|Elektronik|1800000|     1|
|           8| Laptop|Elektronik|7500000|     1|
|          10|Monitor|Elektronik|2500000|     3|
+------------+-------+----------+-------+------+



In [23]:
# Membuat kolom baru

from pyspark.sql.functions import col

df_total = df.withColumn (
    "total",
    col("harga") * col("jumlah")
)

df_total.show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
+------------+---------+----------+-------+------+--------+



In [24]:
# mengurutkan data
df_total.orderBy(
    col("total").desc()
) .show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
+------------+---------+----------+-------+------+--------+



# E.3 Agregasi Data Menggunakan DataFrame API

In [25]:
# Menghitung total pendapatan setiap kategori
from pyspark.sql.functions import sum

df_total.groupBy("Kategori") \
    .agg(sum("total").alias("total_pendapatan")
    ) .show()

+----------+----------------+
|  Kategori|total_pendapatan|
+----------+----------------+
|Elektronik|        36800000|
| Aksesoris|         4650000|
+----------+----------------+



In [26]:
# Menghitung jumlah transaksi setiap kategori
from pyspark.sql.functions import count

df_total.groupBy("kategori") \
    .agg(
        count("*").alias("jumlah_transaksi")
    ).show()

+----------+----------------+
|  kategori|jumlah_transaksi|
+----------+----------------+
|Elektronik|               5|
| Aksesoris|               5|
+----------+----------------+



In [27]:
# Menghitung rata-rata nilai transaksi
from pyspark.sql.functions import avg

df_total.groupBy("kategori") \
    .agg(
        avg("total").alias("rata_rata_transaksi")
    ).show()

+----------+-------------------+
|  kategori|rata_rata_transaksi|
+----------+-------------------+
|Elektronik|          7360000.0|
| Aksesoris|           930000.0|
+----------+-------------------+



In [28]:
# Menghitung statistik keseluruhan
df_total.select(
    sum("total").alias("total_penjualan"),
    avg("total").alias("rata_rata_penjualan"),
    count("*").alias("jumlah_transaksi")
).show()

+---------------+-------------------+----------------+
|total_penjualan|rata_rata_penjualan|jumlah_transaksi|
+---------------+-------------------+----------------+
|       41450000|          4145000.0|              10|
+---------------+-------------------+----------------+



# E.4 Pemrosesan Data Menggunakan Spark SQL

In [29]:
# Membuat Temporary View
df_total.createOrReplaceTempView(
    "transaksi"
)

In [30]:
# Menampilkan seluruh data
spark.sql("""
    SELECT *
    FROM transaksi
""").show()

+------------+---------+----------+-------+------+--------+
|id_transaksi|   produk|  kategori|  harga|jumlah|   total|
+------------+---------+----------+-------+------+--------+
|           1|   Laptop|Elektronik|7500000|     2|15000000|
|           2|    Mouse| Aksesoris| 150000|     5|  750000|
|           3| Keyboard| Aksesoris| 300000|     3|  900000|
|           4|  Monitor|Elektronik|2500000|     2| 5000000|
|           5|Flashdisk| Aksesoris|  80000|    10|  800000|
|           6|  Printer|Elektronik|1800000|     1| 1800000|
|           7|  Headset| Aksesoris| 250000|     4| 1000000|
|           8|   Laptop|Elektronik|7500000|     1| 7500000|
|           9|    Mouse| Aksesoris| 150000|     8| 1200000|
|          10|  Monitor|Elektronik|2500000|     3| 7500000|
+------------+---------+----------+-------+------+--------+



In [31]:
# Melakukan filtering menggunakan SQL
spark.sql("""
    SELECT produk, kategori, harga, jumlah, total
    FROM transaksi
    WHERE total > 1000000
""").show()

+-------+----------+-------+------+--------+
| produk|  kategori|  harga|jumlah|   total|
+-------+----------+-------+------+--------+
| Laptop|Elektronik|7500000|     2|15000000|
|Monitor|Elektronik|2500000|     2| 5000000|
|Printer|Elektronik|1800000|     1| 1800000|
| Laptop|Elektronik|7500000|     1| 7500000|
|  Mouse| Aksesoris| 150000|     8| 1200000|
|Monitor|Elektronik|2500000|     3| 7500000|
+-------+----------+-------+------+--------+



In [32]:
# Melakukan agregasi menggunakan SQL
spark.sql("""
    SELECT kategori,
           SUM(total) AS total_pendapatan,
           COUNT(*) AS jumlah_transaksi,
           AVG(total) AS rata_rata_transaksi
    FROM transaksi
    GROUP BY kategori
""").show()

+----------+----------------+----------------+-------------------+
|  kategori|total_pendapatan|jumlah_transaksi|rata_rata_transaksi|
+----------+----------------+----------------+-------------------+
|Elektronik|        36800000|               5|          7360000.0|
| Aksesoris|         4650000|               5|           930000.0|
+----------+----------------+----------------+-------------------+



In [33]:
# Mengurutkan hasil query
spark.sql("""
    SELECT produk,
           SUM(total) AS total_penjualan
    FROM transaksi
    GROUP BY produk
    ORDER BY total_penjualan DESC
""").show()

+---------+---------------+
|   produk|total_penjualan|
+---------+---------------+
|   Laptop|       22500000|
|  Monitor|       12500000|
|    Mouse|        1950000|
|  Printer|        1800000|
|  Headset|        1000000|
| Keyboard|         900000|
|Flashdisk|         800000|
+---------+---------------+



# F. Eksperimen Lazy Evaluation


In [34]:
# Mendefinisikan transformation
df_filter = df.filter(
    df.harga > 1000000
)

print("Transformation selesai")

# Menampilkan data setelah action dipanggil
df_filter.show()

# Melihat rencana eksekusi Spark
df_filter.explain()

Transformation selesai
+------------+-------+----------+-------+------+
|id_transaksi| produk|  kategori|  harga|jumlah|
+------------+-------+----------+-------+------+
|           1| Laptop|Elektronik|7500000|     2|
|           4|Monitor|Elektronik|2500000|     2|
|           6|Printer|Elektronik|1800000|     1|
|           8| Laptop|Elektronik|7500000|     1|
|          10|Monitor|Elektronik|2500000|     3|
+------------+-------+----------+-------+------+

== Physical Plan ==
*(1) Filter (isnotnull(harga#3L) AND (harga#3L > 1000000))
+- *(1) Scan ExistingRDD[id_transaksi#0L,produk#1,kategori#2,harga#3L,jumlah#4L]




# Tugas

In [35]:
# Tugas 1. Analisis Produk
# Menampilkan lima produk dengan total pendapatan tertinggi menggunakan DataFrame API
df_total.groupBy("produk") \
    .agg(sum("total").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc()) \
    .limit(5) \
    .show()

+-------+----------------+
| produk|total_pendapatan|
+-------+----------------+
| Laptop|        22500000|
|Monitor|        12500000|
|  Mouse|         1950000|
|Printer|         1800000|
|Headset|         1000000|
+-------+----------------+



In [36]:
# Tugas 2. Analisis Kategori
# Menghitung total pendapatan, jumlah transaksi, dan rata-rata jumlah produk terjual setiap kategori
df_total.groupBy("kategori") \
    .agg(
        sum("total").alias("total_pendapatan"),
        count("*").alias("jumlah_transaksi"),
        avg("jumlah").alias("rata_rata_jumlah_terjual")
    ).show()

+----------+----------------+----------------+------------------------+
|  kategori|total_pendapatan|jumlah_transaksi|rata_rata_jumlah_terjual|
+----------+----------------+----------------+------------------------+
|Elektronik|        36800000|               5|                     1.8|
| Aksesoris|         4650000|               5|                     6.0|
+----------+----------------+----------------+------------------------+



In [37]:
# Tugas 3. Filtering Data
# Menampilkan transaksi dengan harga > Rp100.000 dan jumlah pembelian minimal 3 unit
df_total.filter(
    (col("harga") > 100000) & (col("jumlah") >= 3)
).show()

+------------+--------+----------+-------+------+-------+
|id_transaksi|  produk|  kategori|  harga|jumlah|  total|
+------------+--------+----------+-------+------+-------+
|           2|   Mouse| Aksesoris| 150000|     5| 750000|
|           3|Keyboard| Aksesoris| 300000|     3| 900000|
|           7| Headset| Aksesoris| 250000|     4|1000000|
|           9|   Mouse| Aksesoris| 150000|     8|1200000|
|          10| Monitor|Elektronik|2500000|     3|7500000|
+------------+--------+----------+-------+------+-------+



In [38]:
# Tugas 4. Spark SQL
# Query SQL untuk menampilkan nama produk dan total pendapatan setiap produk, urut terbesar
df_total.createOrReplaceTempView("transaksi")

spark.sql("""
    SELECT produk,
           SUM(total) AS total_pendapatan
    FROM transaksi
    GROUP BY produk
    ORDER BY total_pendapatan DESC
""").show()

+---------+----------------+
|   produk|total_pendapatan|
+---------+----------------+
|   Laptop|        22500000|
|  Monitor|        12500000|
|    Mouse|         1950000|
|  Printer|         1800000|
|  Headset|         1000000|
| Keyboard|          900000|
|Flashdisk|          800000|
+---------+----------------+

